# Laboratorio 1 — Ejercicio 6
## Tasa de interés de un préstamo

La cuota mensual nivelada de un préstamo se calcula mediante

$$
A=P\frac{r(1+r)^n}{(1+r)^n-1},
$$

donde $P$ es el monto prestado, $r$ la tasa mensual y $n$ el número de pagos. Para un préstamo de USD 50,000, pagadero en 36 meses con cuotas de USD 1,650, se debe hallar $r$ y la tasa nominal anual $12r$ con tolerancia $\varepsilon=10^{-7}$.

## Resumen del ejercicio

La tasa desconocida se obtiene convirtiendo la fórmula financiera en una ecuación de raíz. Se aplica Bisección sobre un intervalo de tasas mensuales positivas que presenta un cambio de signo, registrando la aproximación, su cota de error y el residuo en cada iteración.

El resultado se encuentra cerca de $r=0.00962537$, es decir, aproximadamente $0.962537\%$ mensual. La tasa nominal anual es cercana a $11.550439\%$. Finalmente, se sustituye la tasa obtenida en la fórmula original para comprobar que reproduce la cuota mensual indicada.

## Objetivo y método

Se utiliza Bisección porque la función de la cuota es continua para $r>0$ y puede encerrarse entre dos tasas con residuos de signos opuestos. A diferencia de un método abierto, no requiere derivar la expresión financiera y proporciona una cota explícita del error mediante la mitad del ancho del intervalo.

## 1. Formulación como búsqueda de raíz

Al trasladar la cuota conocida al lado izquierdo se obtiene

$$
f(r)=50000\frac{r(1+r)^{36}}{(1+r)^{36}-1}-1650.
$$

Resolver $f(r)=0$ equivale a encontrar la tasa para la cual la cuota calculada coincide con USD 1,650. Se trabajará únicamente con $r>0$, evitando el valor $r=0$, donde la expresión escrita presenta la forma indeterminada $0/0$.

In [1]:
PRESTAMO = 50_000.0
PAGOS = 36
CUOTA_OBJETIVO = 1_650.0

def cuota_mensual(r):
    factor = (1 + r)**PAGOS
    return PRESTAMO * r * factor / (factor - 1)

def f(r):
    return cuota_mensual(r) - CUOTA_OBJETIVO

### Explicación de esta parte

Las constantes conservan los datos financieros separados de la implementación. `cuota_mensual(r)` representa la fórmula original y `f(r)` calcula el residuo respecto de la cuota objetivo. Un residuo negativo indica que la tasa produce una cuota menor que USD 1,650; uno positivo indica una cuota mayor.

## 2. Selección del intervalo inicial

Se prueban las tasas mensuales $0.5\%=0.005$ y $1\%=0.01$. Ambas son positivas, por lo que la fórmula puede evaluarse directamente sin tratar el caso singular $r=0$.

In [2]:
for r in (0.005, 0.01):
    print(
        f"r = {r:.3%}  "
        f"cuota = ${cuota_mensual(r):,.6f}  "
        f"f(r) = {f(r): .6f}"
    )

r = 0.500%  cuota = $1,521.096873  f(r) = -128.903127
r = 1.000%  cuota = $1,660.715491  f(r) =  10.715491


### Interpretación del intervalo

En $r=0.005$ el residuo es negativo y en $r=0.01$ es positivo. Como $f$ es continua entre ambas tasas, existe al menos una raíz en $[0.005,0.01]$. Además, la cuota de un préstamo de pagos nivelados aumenta con la tasa positiva, de modo que la solución encerrada es única.

## 3. Implementación de Bisección

En cada paso se conserva el subintervalo que mantiene el cambio de signo. El error se define como

$$
e_k=\frac{|b_k-a_k|}{2},
$$

que es una cota para la distancia entre el punto medio y la raíz. La tolerancia requerida es $\varepsilon=10^{-7}$. Para que los porcentajes finales se muestren con cifras estables, el cálculo se continúa hasta $e_k<10^{-10}$, un criterio más estricto que también satisface el solicitado. Se dispone de un máximo de 100 iteraciones como salvaguarda.

In [3]:
def biseccion(funcion, a, b, tolerancia=1e-7, max_iteraciones=100):
    fa = funcion(a)
    fb = funcion(b)

    if fa == 0:
        return a, []
    if fb == 0:
        return b, []
    if fa * fb > 0:
        raise ValueError("El intervalo no presenta un cambio de signo.")

    historial = []

    for iteracion in range(1, max_iteraciones + 1):
        punto_evaluado = (a + b) / 2
        fp = funcion(punto_evaluado)

        if fa * fp <= 0:
            b = punto_evaluado
            fb = fp
        else:
            a = punto_evaluado
            fa = fp

        aproximacion = (a + b) / 2
        error = abs(b - a) / 2
        residuo = abs(funcion(aproximacion))
        historial.append(
            (iteracion, a, b, aproximacion, error, residuo)
        )

        if error < tolerancia:
            return aproximacion, historial

    raise RuntimeError("Bisección no convergió en el máximo de iteraciones.")

### Explicación del algoritmo implementado

La función valida primero la precondición de cambio de signo. En cada iteración actualiza uno de los extremos y registra el nuevo punto medio. El historial conserva tanto la cota de error en la tasa como el residuo monetario, que mide la diferencia absoluta entre la cuota producida y los USD 1,650 solicitados.

La cota de Bisección es el criterio formal de parada; el residuo se incluye como comprobación adicional y no sustituye la tolerancia especificada para $r$.

## 4. Cálculo de la tasa

La tabla muestra las primeras cinco y las últimas cinco iteraciones para conservar una salida compacta. Las iteraciones intermedias siguen la misma reducción del intervalo.

In [4]:
tolerancia_requerida = 1e-7
tolerancia_calculo = 1e-10
raiz, historial = biseccion(f, 0.005, 0.01, tolerancia_calculo)

print(
    f"{'k':>2} {'a_k':>13} {'b_k':>13} "
    f"{'r_k':>13} {'error':>12} {'residuo':>12}"
)
filas = historial if len(historial) <= 10 else historial[:5] + historial[-5:]
for k, a, b, r, error, residuo in filas:
    print(
        f"{k:2d} {a:13.10f} {b:13.10f} "
        f"{r:13.10f} {error:12.3e} {residuo:12.3e}"
    )

print()
print(f"Tasa mensual aproximada: r = {raiz:.10f}")
print(f"Iteraciones necesarias: {len(historial)}")
print(f"Cota final del error: {historial[-1][4]:.3e}")

 k           a_k           b_k           r_k        error      residuo
 1  0.0075000000  0.0100000000  0.0087500000    1.250e-03    2.488e+01
 2  0.0087500000  0.0100000000  0.0093750000    6.250e-04    7.138e+00
 3  0.0093750000  0.0100000000  0.0096875000    3.125e-04    1.774e+00
 4  0.0093750000  0.0096875000  0.0095312500    1.563e-04    2.686e+00
 5  0.0095312500  0.0096875000  0.0096093750    7.813e-05    4.565e-01
21  0.0096253657  0.0096253681  0.0096253669    1.192e-09    3.321e-05
22  0.0096253657  0.0096253669  0.0096253663    5.960e-10    1.619e-05
23  0.0096253657  0.0096253663  0.0096253660    2.980e-10    7.685e-06
24  0.0096253657  0.0096253660  0.0096253659    1.490e-10    3.431e-06
25  0.0096253657  0.0096253659  0.0096253658    7.451e-11    1.304e-06

Tasa mensual aproximada: r = 0.0096253658
Iteraciones necesarias: 25
Cota final del error: 7.451e-11


### Interpretación de la convergencia

El intervalo se reduce a la mitad en cada iteración y conserva siempre la raíz. La última cota queda por debajo de $10^{-10}$ y, por tanto, también por debajo de la tolerancia requerida $10^{-7}$. Las iteraciones adicionales permiten mostrar de forma estable los porcentajes finales sin cambiar el método.

El valor obtenido está alrededor de $r=0.00962537$. Las últimas cifras pueden variar dentro de la cota de Bisección, pero el intervalo final certifica el error máximo en la tasa sin depender únicamente de que el residuo monetario sea pequeño.

## 5. Conversión y verificación financiera

La tasa mensual se expresa también como porcentaje. La tasa nominal anual solicitada se calcula como $12r$; no debe confundirse con la tasa efectiva anual $(1+r)^{12}-1$, que no fue requerida.

Después se vuelve a evaluar la fórmula original para comprobar la cuota producida por la aproximación.

In [5]:
tasa_nominal_anual = 12 * raiz
cuota_verificada = cuota_mensual(raiz)

print(f"Tasa mensual:        {raiz:.10f} = {100 * raiz:.6f}%")
print(
    f"Tasa nominal anual: {tasa_nominal_anual:.10f} "
    f"= {100 * tasa_nominal_anual:.6f}%"
)
print(f"Cuota calculada:     ${cuota_verificada:,.6f}")
print(f"Cuota objetivo:      ${CUOTA_OBJETIVO:,.6f}")
print(f"Diferencia absoluta: ${abs(cuota_verificada - CUOTA_OBJETIVO):.6f}")

Tasa mensual:        0.0096253658 = 0.962537%
Tasa nominal anual: 0.1155043897 = 11.550439%
Cuota calculada:     $1,650.000001
Cuota objetivo:      $1,650.000000
Diferencia absoluta: $0.000001


### Interpretación de la verificación

La tasa mensual es aproximadamente $0.962537\%$ y su equivalente nominal anual es cercano a $11.550439\%$. Al sustituir la aproximación en la ecuación del préstamo, la cuota calculada queda prácticamente en USD 1,650; la pequeña diferencia corresponde a la tolerancia aplicada sobre $r$.

La multiplicación por 12 es una conversión nominal. No incorpora capitalización mensual y, por ello, no se reemplaza por la fórmula de tasa efectiva anual.

## Resultado

La búsqueda de raíz mediante Bisección produce una tasa mensual cercana a

$$
\boxed{r\approx0.00962537}
\qquad\text{o}\qquad
\boxed{0.962537\%\text{ mensual}}.
$$

La tasa nominal anual solicitada es

$$
\boxed{12r\approx0.11550439}
\qquad\text{o}\qquad
\boxed{11.550439\%\text{ anual nominal}}.
$$

### Conclusión

La fórmula de cuota nivelada se convirtió correctamente en un problema de búsqueda de raíz. El cambio de signo garantiza que el intervalo inicial contiene la solución y la cota propia de Bisección certifica la tolerancia de $10^{-7}$. La sustitución final conecta el resultado numérico con su significado financiero: la tasa obtenida reproduce la cuota mensual de USD 1,650 para el préstamo de USD 50,000 a 36 meses.